In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import pandas as pd
import os

# ============================================
# 1. Custom Dataset for DataFrame-based inputs
# ============================================

class ImageDFDataset(Dataset):
    def __init__(self, df, label_to_idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = '../data' + self.df.loc[idx, "image_path"]
        label_str = self.df.loc[idx, "label"]
        label = self.label_to_idx[label_str]

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


# ============================================
# 2. Build label mapping
# ============================================

def build_label_mapping(df_train):
    classes = sorted(df_train["label"].unique())
    label_to_idx = {c: i for i, c in enumerate(classes)}
    idx_to_label = {i: c for c, i in label_to_idx.items()}
    return label_to_idx, idx_to_label


# ============================================
# 3. Transforms
# ============================================

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.5),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(144),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# ============================================
# 4. Create datasets & loaders from dataframe
# ============================================

def make_dataloaders(df_train, df_val, batch_size=32):

    label_to_idx, idx_to_label = build_label_mapping(df_train)
    num_classes = len(label_to_idx)

    train_dataset = ImageDFDataset(df_train, label_to_idx, transform=train_transform)
    val_dataset   = ImageDFDataset(df_val, label_to_idx, transform=val_transform)

    # ---- Balanced sampler (important for ~20 images/class) ----
    class_counts = df_train["label"].value_counts().sort_index()
    class_weights = 1.0 / torch.tensor(class_counts.tolist(), dtype=torch.float)

    sample_weights = [
        class_weights[label_to_idx[label]]
        for label in df_train["label"]
    ]

    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, num_classes, label_to_idx, idx_to_label


# ============================================
# 5. Build ResNet-18 (train from scratch)
# ============================================

def build_resnet18(num_classes):
    model = models.resnet18(weights=None)   # NOT pretrained

    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(512, num_classes)
    )

    return model


def build_squeezenet(num_classes):
    model = models.squeezenet1_1(weights=None)  # no pretrained weights

    # Replace classifier
    model.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=1)
    model.num_classes = num_classes

    return model


# ============================================
# 6. Training loop (clean version)
# ============================================

def train_model(model, train_loader, val_loader, epochs=100, lr=1e-3):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        val_loss = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        print(f"Epoch {epoch+1}/{epochs} "
              f"| Train Loss: {running_loss/len(train_loader):.4f} "
              f"| Val Loss: {val_loss:.4f}")


# ============================================
# 7. Validation
# ============================================

@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()

    return total_loss / len(loader)




In [8]:
import torch
import torch.nn as nn

class ImageEmbeddingModel(nn.Module):
    def __init__(self, img_size=128):  # <-- default to 128
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )

        # Compute flat size dynamically
        with torch.no_grad():
            x = torch.randn(1, 3, img_size, img_size)
            x = self.backbone(x)
            flat_size = x.numel() // x.shape[0]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 256),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 200)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))


def build_custom_cnn():
    return ImageEmbeddingModel(img_size=128)


In [9]:
from sklearn.model_selection import train_test_split
# ============================================
# 8. Example usage
# ============================================
df = pd.read_csv("../data/train_images.csv")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_loader, val_loader, num_classes, label_to_idx, idx_to_label = \
    make_dataloaders(train_df, val_df)

model = build_custom_cnn()

In [10]:
train_model(model, train_loader, val_loader, epochs=120, lr=1e-3)

Epoch 1/120 | Train Loss: 5.3017 | Val Loss: 5.2994
Epoch 2/120 | Train Loss: 5.2950 | Val Loss: 5.2737
Epoch 3/120 | Train Loss: 5.2460 | Val Loss: 5.2189
Epoch 4/120 | Train Loss: 5.1941 | Val Loss: 5.1076
Epoch 5/120 | Train Loss: 5.1341 | Val Loss: 5.0551
Epoch 6/120 | Train Loss: 5.0529 | Val Loss: 5.0148
Epoch 7/120 | Train Loss: 5.0375 | Val Loss: 4.9888
Epoch 8/120 | Train Loss: 4.9840 | Val Loss: 4.9576
Epoch 9/120 | Train Loss: 4.9253 | Val Loss: 4.9219
Epoch 10/120 | Train Loss: 4.9078 | Val Loss: 4.8962
Epoch 11/120 | Train Loss: 4.8143 | Val Loss: 4.8620
Epoch 12/120 | Train Loss: 4.8189 | Val Loss: 4.8176
Epoch 13/120 | Train Loss: 4.7488 | Val Loss: 4.7849
Epoch 14/120 | Train Loss: 4.7564 | Val Loss: 4.7669
Epoch 15/120 | Train Loss: 4.6665 | Val Loss: 4.7672
Epoch 16/120 | Train Loss: 4.6346 | Val Loss: 4.7214
Epoch 17/120 | Train Loss: 4.6094 | Val Loss: 4.6957
Epoch 18/120 | Train Loss: 4.5284 | Val Loss: 4.6471
Epoch 19/120 | Train Loss: 4.5014 | Val Loss: 4.6970
Ep

KeyboardInterrupt: 

In [11]:
from sklearn.metrics import accuracy_score
val_preds = []
val_labels = []
model.eval()

with torch.no_grad():
    for inputs, labels in val_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        val_preds.extend(predicted.numpy())
        val_labels.extend(labels.numpy())

accuracy = accuracy_score(val_labels, val_preds)

print(accuracy)

0.15267175572519084


In [12]:
test_df = pd.read_csv("../data/test_images_path.csv")
test_dataset   = ImageDFDataset(test_df, label_to_idx, transform=val_transform)
test_loader   = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [13]:
model.eval()

ImageEmbeddingModel(
  (backbone): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=16384, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=200, bias=

In [14]:
all_ids = test_df["id"].tolist()
all_preds = []

In [15]:
with torch.no_grad():
    for inputs, _ in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.numpy())

In [16]:
predicted_labels = [idx_to_label[i] for i in all_preds]

In [17]:
output_df = pd.DataFrame({
    "id": all_ids,
    "label": predicted_labels
})

output_df.to_csv("test_predictions.csv", index=False)
print("Saved test_predictions.csv!")

Saved test_predictions.csv!
